# Practice Lab: Open-Source Fine-Tuning (Lesson 9.4)

Module 9 · 8 exercises · LoRA math + VRAM + frameworks + DPO + GRPO + multi-LoRA

Exercises 1, 2, 5 run locally (pure Python).


# Lesson 9.4: Open-Source Fine-Tuning — HF Trainer & Unsloth

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 5 run **locally** (pure Python). Ex 3, 4, 6, 7, 8 are design/architecture.


In [ ]:
import numpy as np


---
## Exercise 1 (Easy): LoRA Parameter Calculator


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
def lora_params(hidden, rank, n_modules, n_layers):
    return 2 * hidden * rank * n_modules * n_layers

h, L, total = 4096, 32, 6_738_000_000

configs = {"attn-only (4)": 4, "all-linear (7)": 7}

print("LoRA Parameter Calculator (Llama 7B):")
for cfg, n_mod in configs.items():
    print(f"\n  {cfg}:")
    print(f"  {'Rank':<8} {'Params':>12} {'%':>8} {'Size':>10}")
    print(f"  {'-'*42}")
    for r in [4, 8, 16, 32, 64]:
        lp = lora_params(h, r, n_mod, L)
        print(f"  r={r:<5} {lp:>12,} {lp/total*100:>7.2f}% {lp*2/1e6:>8.1f} MB")

print(f"\nProduction: r=16, all-linear, DoRA=True, rsLoRA=True")
print(f"Alpha=2*rank (sweet spot). Lower rank + MORE modules > high rank + few.")

print(f"\nAlpha scaling:")
for r in [8, 16, 32]: print(f"  r={r}, alpha={r*2}, scale={r*2/r:.1f}")


</details>


---
## Exercise 2 (Easy): QLoRA Memory Estimator


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
def vram(p, method="full"):
    if method == "full":
        return round(p*2/1e9 + p*2/1e9 + p*4*2/1e9 + 2, 1)  # model+grad+opt+overhead
    elif method == "lora":
        return round(p*2/1e9 + 3 + 2, 1)  # frozen fp16 + adapter + overhead
    elif method == "qlora":
        return round(p*0.5/1e9 + 2 + 2, 1)  # 4-bit + adapter + overhead
    elif method == "qlora_unsloth":
        return round(vram(p, "qlora") * 0.6, 1)  # 40% savings

models = [("7B", 7e9), ("13B", 13e9), ("70B", 70e9)]
gpus = [("T4 (free)", 16), ("L4/4090", 24), ("A100 40GB", 40), ("A100 80GB", 80)]

print("QLoRA Memory Estimator:")
print(f"\n{'Model':<8} {'Full':>10} {'LoRA':>10} {'QLoRA':>10} {'QLoRA+US':>10}")
print("-"*52)
for name, p in models:
    print(f"  {name:<6}", end="")
    for m in ["full","lora","qlora","qlora_unsloth"]:
        print(f" {vram(p,m):>8.1f}GB", end="")
    print()

print(f"\nGPU Fit (QLoRA+Unsloth):")
for name, p in models:
    v = vram(p, "qlora_unsloth")
    fit = next((g for g,gv in gpus if v<=gv), "Multi-GPU")
    print(f"  {name}: {v}GB -> {fit}")

print(f"\nFull 7B=120GB -> QLoRA+Unsloth=3.5GB (34x reduction)")


</details>


---
## Exercise 5 (Medium): Framework Comparison


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
fws = [
    ("Unsloth+TRL", "25K", "3.2hr", "Pro only", "2x speed, 70% less VRAM"),
    ("Axolotl", "11.5K", "5.8hr", "FSDP2/DS", "Multi-GPU production"),
    ("LLaMA-Factory", "45K", "3.4hr", "Yes", "Zero-code Web UI, 100+ models"),
    ("torchtune", "5.7K", "4.7hr", "FSDP2", "Pure PyTorch, max control"),
]

print("Training Framework Comparison:")
print(f"{'Framework':<16} {'Stars':<8} {'Speed':<8} {'MultiGPU':<12} {'Best For'}")
print("-"*65)
for name,stars,speed,mgpu,best in fws:
    print(f"  {name:<14} {stars:<8} {speed:<8} {mgpu:<12} {best}")

print(f"\nBenchmark (Llama-3.1 8B QLoRA on A100):")
for name,_,speed,_,_ in sorted(fws, key=lambda x: float(x[2].replace("hr",""))):
    print(f"  {name:<16} {speed}")

print(f"\nDecision Guide:")
decisions = [("Single GPU speed","Unsloth+TRL"),("Multi-GPU prod","Axolotl"),
             ("Zero-code","LLaMA-Factory"),("Research/control","torchtune"),
             ("Indian languages","Unsloth+TRL"),("GRPO reasoning","Unsloth (80% less VRAM)")]
for s,r in decisions: print(f"  {s:<28} -> {r}")


</details>


---
## Exercise 3 (Medium): Unsloth SFT Pipeline
Design/architecture. See practice-lab-9_4.html.


In [ ]:
# Exercise 3: Unsloth SFT Pipeline
pass


---
## Exercise 4 (Medium): DPO Alignment Pipeline
Design/architecture. See practice-lab-9_4.html.


In [ ]:
# Exercise 4: DPO Alignment Pipeline
pass


---
## Exercise 6 (Challenge): GRPO Reasoning Training
Design/architecture. See practice-lab-9_4.html.


In [ ]:
# Exercise 6: GRPO Reasoning Training
pass


---
## Exercise 7 (Challenge): Multi-LoRA Deployment
Design/architecture. See practice-lab-9_4.html.


In [ ]:
# Exercise 7: Multi-LoRA Deployment
pass


---
## Exercise 8 (Challenge): Full Production Pipeline
Design/architecture. See practice-lab-9_4.html.


In [ ]:
# Exercise 8: Full Production Pipeline
pass
